In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))


## Merge game databases

### Steps
- pick a main dataset
- choose datasets to be merged
- SourceConfig-s with platform, column mappings
- merge

In [2]:
%pip install python-Levenshtein

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import pickle
from utils.merge_pipeline import SourceConfig, run_merge_pipeline
from utils.gamelist_parser import load_all_gamelists

merged_write_location = '../output/merged_df.pkl'

## Define main dataset config


### Launchbox

Lets choose the Launchbox dataset as our main to which we will merge other datasets

In [4]:
launchbox_column_map={
    'Name': 'name',
    'ReleaseYear': 'release_year',
    'Overview': 'summary',
    'MaxPlayers': 'players',
    'Cooperative': 'cooperative',
    'Platform': 'platform',
    'CommunityRating': 'user_rating',
    'Genres': 'genres',
    'Developer': 'developer',
    'Publisher': 'publisher'
}

main_config = SourceConfig(
    name='launchbox',
    path='../csv/launchbox.csv',
    rename_map=launchbox_column_map,
    transforms={
        'user_rating': lambda s: pd.to_numeric(s, errors='coerce') * 2,
    }
 )

## Datasets to be merged

### all_games

In [5]:
all_games_column_map={
    'meta_score': 'rating',
    'user_review': 'user_rating'
}

all_games_config = SourceConfig(
    name='all_games',
    path='../csv/all_games.csv',
    rename_map=all_games_column_map
)

### GameTDB

In [6]:
gametdb_column_map={
    'gametdb_platform': 'platform',
    'gametdb_title_en': 'name',
    'gametdb_synopsis_en': 'summary',
    'gametdb_developer': 'developer',
    'gametdb_publisher': 'publisher',
    'gametdb_date': 'release_date',
    'gametdb_players': 'players',
    'gametdb_genre': 'genres'
}

gametdb_config = SourceConfig(
    name='gametdb',
    path='../csv/games_on_gametdb.csv',
    rename_map=gametdb_column_map
)

### DAT DOS dataset

In [7]:
dat_dos_column_map={
    'gametitle_mame': 'name', 
    'developer_mame': 'developer', 
    'year_mame': 'release_date', 
    'genre_mame': 'genres',
    'description_mame': 'summary',
    'publisher_mame': 'publisher'
}

dat_dos_config = SourceConfig(
    name='dat_dos',
    path='../csv/dat_database_dos.csv',
    rename_map=dat_dos_column_map,
    constants={'platform': 'MS-DOS'}
)

### DAT MAME

In [8]:
dat_mame_column_map={
    'gametitle_mame': 'name',
    'filename_mame': 'filename',
    'manufacturer_mame': 'developer',
    'year_mame': 'release_date',
    'genre_mame': 'genres'
}

dat_mame_config = SourceConfig(
    name='dat_mame',
    path='../csv/dat_database_mame.csv',
    read_csv_kwargs={'low_memory': False},
    loader=lambda: (
        pd.read_csv('../csv/dat_database_mame.csv', low_memory=False)
        .query("systemname_mame == 'MAME 0.240 (Arcade).dat'")
    ),
    rename_map=dat_mame_column_map,
    constants={'platform': 'Arcade'}
)

### Recalbox

In [9]:
recalbox_column_map={
    'nomjeu_rom': 'name',
    'fichier_rom': 'filename',
    'description_rom': 'summary',
    'rating_rom': 'user_rating',
    'annee_rom': 'release_date',
    'developer_rom': 'developer',
    'publisher_rom': 'publisher',
    'genre_rom': 'genres',
    'players_rom': 'players'
}

recalbox_config = SourceConfig(
    name='recalbox',
    path='../csv/recalbox_gamelist.csv',
    rename_map=recalbox_column_map,
    constants={'platform': 'Nintendo Entertainment System'},
    transforms={
        'user_rating': lambda s: pd.to_numeric(s, errors='coerce') * 10
    }
)

### game dataset cleaned

In [10]:
game_dataset_rename_map = {
    'platforms': 'platform',
    'main_developers': 'developer',
    'publishers': 'publisher',
    'aggregated_rating': 'user_rating'
}

game_dataset_config = SourceConfig(
    name='game_dataset',
    path='../csv/game_dataset_cleaned.csv',
    rename_map=game_dataset_rename_map,
    # platform_map=game_dataset_platform_map,
    transforms={
        'user_rating': lambda s: pd.to_numeric(s, errors='coerce') / 10
    }
)

### gamelist.xml files

In [11]:
gamelist_config = SourceConfig(
    name="gamelist",
    loader=lambda: load_all_gamelists(lists_dir="lists")
)

### The final merge

In [12]:
# source_configs list with all sources to merge
source_configs = [all_games_config, gametdb_config, dat_dos_config, dat_mame_config, recalbox_config, game_dataset_config, gamelist_config]

# Call run_merge_pipeline() to orchestrate the merge
merged_df = run_merge_pipeline(
    main_config=main_config,
    source_configs=source_configs
    # use this to collapse platforms for the same game into a list.
    # collapse_platforms=True
)

print(f'Final merged rows: {len(merged_df)}')
print(f'Unique platforms: {merged_df["platform"].nunique()}')
merged_df.head()


  [pre-merge dedup] launchbox: removed 1721 duplicates (164836 -> 163115)
  [pre-merge dedup] all_games: removed 113 duplicates (18800 -> 18687)
main size: 171939, all_games size: 18687, new games: 8824
  [pre-merge dedup] gametdb: removed 15863 duplicates (52940 -> 37077)
main size: 195728, gametdb size: 37077, new games: 23789
  [pre-merge dedup] dat_dos: removed 432 duplicates (7482 -> 7050)
main size: 197715, dat_dos size: 7050, new games: 1987
  [pre-merge dedup] dat_mame: removed 232 duplicates (36604 -> 36372)
main size: 231794, dat_mame size: 36372, new games: 34079
  [pre-merge dedup] recalbox: removed 19 duplicates (535 -> 516)
main size: 232189, recalbox size: 516, new games: 395
Info: name '\\\//\\/\\\///' cleaned to empty match key, using raw fallback key
Info: name ':)' cleaned to empty match key, using raw fallback key
Info: name ':)' cleaned to empty match key, using raw fallback key
Info: name ':)' cleaned to empty match key, using raw fallback key
Info: name '||[}}}°.

,platform,name,filename,summary,release_date,release_year,genres,developer,publisher,players,cooperative,rating,user_rating,version
0,Sony PlayStation,&: Sora no Mukou de Saki Masuyou ni,NaN,'&' - Sora no Mukou de Sakimasu you ni is the ...,NaT,<NA>,Visual Novel,Akatsuki Works,Spike Chunsoft,1.0,False,<NA>,7.145833,NaN
4,Web Browser,& in the War I Find You,NaN,When the world has gone crazy - please hold on...,NaT,<NA>,"Indie, Visual Novel",Lawrence Marable,Lawrence Marable,NaN,<NA>,<NA>,NaN,NaN
5,Windows,&&,NaN,A bizarre role-playing game in an alien landsc...,2015-12-31,<NA>,"Adventure, Role-playing (RPG)",Olvin Weplanis,Olvin Weplanis,NaN,<NA>,<NA>,NaN,NaN
6,Google Android,&0,NaN,Detective otome game.,2022-05-30,<NA>,"Simulator, Visual Novel",Coly,Coly,NaN,<NA>,<NA>,NaN,NaN
8,Commodore 64,'43: One Year After,NaN,'43 - One Year After is a vertical-scrolling s...,NaT,1986,Shooter,Greve Graphics,Action Software,1.0,False,<NA>,7.750000,NaN


In [13]:
# platforms
merged_df['platform'].value_counts()

platform
Windows                      133904
Arcade                        41486
Mac                           23329
Nintendo Switch               19933
Linux                         15811
                              ...  
Elektor TV Games Computer         1
AY-3-8605                         1
Donner Model 30                   1
Virtual Console                   1
Namco System 22                   1
Name: count, Length: 276, dtype: int64

In [14]:
print(f'Unique platforms: {merged_df["platform"].nunique()}')
sorted(merged_df['platform'].unique().tolist())

Unique platforms: 276


['1292 Advanced Programmable Video System',
 '3DO',
 'APF Imagination Machine',
 'AY-3-8500',
 'AY-3-8603',
 'AY-3-8605',
 'AY-3-8606',
 'AY-3-8607',
 'AY-3-8610',
 'AY-3-8760',
 'Aamber Pegasus',
 'Acorn Archimedes',
 'Acorn Atom',
 'Acorn Electron',
 'Advanced Pico Beena',
 'AirConsole',
 'Amazon Fire TV',
 'Amstrad CPC',
 'Amstrad GX4000',
 'Analogue electronics',
 'Apogee BK-01',
 'Apple II',
 'Apple IIGS',
 'Arcade',
 'Arcadia 2001',
 'Arduboy',
 'Atari 2600',
 'Atari 5200',
 'Atari 7800 ProSystem',
 'Atari 800',
 'Atari Jaguar',
 'Atari Jaguar CD',
 'Atari Lynx',
 'Atari ST',
 'Atari XE',
 'Atomiswave',
 'BBC Microcomputer System',
 'Bally Astrocade',
 'Bandai Super Vision 8000',
 'Bandai WonderSwan',
 'Bandai WonderSwan Color',
 'BlackBerry OS',
 'Blu-ray Player',
 'CDC Cyber 70',
 'Call-A-Computer time-shared mainframe computer system',
 'Camputers Lynx',
 'Card-e Reader',
 'Casio Loopy',
 'Casio PV-1000',
 'Coleco ADAM',
 'ColecoVision',
 'Commodore 128',
 'Commodore 64',
 'Co

In [15]:
# write to file
with open(merged_write_location, 'wb') as f:
    pickle.dump(merged_df, f)

In [16]:
from datetime import datetime
from utils.merge_pipeline import CANONICAL_SCHEMA
from utils.csv_export import write_to_csv

merged_df['version'] = datetime.utcnow().isoformat()
write_to_csv(merged_df, Path('../output/raw_games.csv'), CANONICAL_SCHEMA)
print(f"Raw merged CSV saved to output/raw_games.csv ({len(merged_df)} rows)")

/tmp/ipykernel_13287/2411244175.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  merged_df['version'] = datetime.utcnow().isoformat()


Raw merged CSV saved to output/raw_games.csv (462785 rows)
